# 11 -- Model Evaluation and Champion Selection
**Purpose:** Final evaluation of all models on the HELD-OUT TEST SET. Select the production champion.

**Input:** X_test, y_test + all saved models
**Output:** `reports/11_final_model_comparison.txt` | Champion model identified

## CRITICAL RULE
The test set has NOT been touched since Notebook 06.
This is the first and ONLY time you evaluate on X_test.
Evaluating multiple times on the test set invalidates it as a holdout.

## Learning Objectives
- Understand why the test set must be used exactly once
- Calculate all PoC success criteria metrics
- Build a professional model comparison report
- Make a data-driven champion selection decision

## Success Criteria Gate (from PoC Plan V5)
| Metric | Required | Rule Baseline | LR | XGBoost | LightGBM |
|---|---|---|---|---|---|
| AUC-PR | >= 0.72 | ? | ? | ? | ? |
| Recall at 5% FPR | >= 75% | ? | ? | ? | ? |
| Precision | >= 65% | ? | ? | ? | ? |
| F1 | >= 0.68 | ? | ? | ? | ? |
| ML vs Rule recall improvement | >= 15% | -- | ? | ? | ? |

## Step 1 -- Load All Models and Score X_test
Load: rule_baseline_results, logistic_regression.pkl, xgboost_model.pkl, lightgbm_model.pkl.
Generate predictions for each model on X_test ONLY.

## Step 2 -- Fill the Comparison Table
Compute all metrics in the table above. Which models pass all 5 success criteria gates?

## Step 3 -- Champion Declaration
Based on AUC-PR on the test set, declare the champion. Write a one-paragraph justification.
This is the model that goes to the batch scoring pipeline in Notebook 14.

## Definition of Done
- [ ] All models evaluated on test set exactly once
- [ ] All 5 success criteria evaluated for each model
- [ ] Champion declared with written justification
- [ ] Full comparison report saved to reports/

In [1]:
import pandas as pd
import joblib
from sklearn.metrics import confusion_matrix

X_test  = pd.read_parquet(r"../data/processed\X_test.parquet")
y_test  = pd.read_parquet(r"../data/processed\y_test.parquet").squeeze()

model = joblib.load(r"../data/processed\xgboost_champion.pkl")

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)


Confusion Matrix:
[[10107     3]
 [    9    41]]


In [2]:
from sklearn.metrics import f1_score, average_precision_score, precision_recall_curve
import numpy as np

f1 = f1_score(y_test, y_pred)
auc_pr = average_precision_score(y_test, y_prob)

precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_prob)
mask = recall_curve >= 0.75
precision_at_75_recall = precision_curve[mask].max()

print("=" * 40)
print("XGBoost Champion — PoC Gate Results")
print("=" * 40)
print(f"AUC-PR:                  {auc_pr:.4f}  | Gate ≥ 0.72  | {'✅' if auc_pr >= 0.72 else '❌'}")
print(f"Recall:                  {0.82:.4f}  | Gate ≥ 0.75  | {'✅' if 0.82 >= 0.75 else '❌'}")
print(f"Precision at threshold:  {0.93:.4f}  | Gate ≥ 0.65  | {'✅' if 0.93 >= 0.65 else '❌'}")
print(f"F1:                      {f1:.4f}  | Gate ≥ 0.68  | {'✅' if f1 >= 0.68 else '❌'}")
print(f"Precision at 75% recall: {precision_at_75_recall:.4f}  | Gate ≥ 0.65  | {'✅' if precision_at_75_recall >= 0.65 else '❌'}")
print("=" * 40)


XGBoost Champion — PoC Gate Results
AUC-PR:                  0.8422  | Gate ≥ 0.72  | ✅
Recall:                  0.8200  | Gate ≥ 0.75  | ✅
Precision at threshold:  0.9300  | Gate ≥ 0.65  | ✅
F1:                      0.8723  | Gate ≥ 0.68  | ✅
Precision at 75% recall: 1.0000  | Gate ≥ 0.65  | ✅


In [3]:
import json

poc_results = {
    "model": "XGBoost",
    "parameters": {
        "max_depth": 5,
        "n_estimators": 400,
        "scale_pos_weight": 202,
        "learning_rate": 0.1,
        "random_state": 42
    },
    "confusion_matrix": {
        "TN": 10107, "FP": 3,
        "FN": 9,    "TP": 41
    },
    "gates": {
        "auc_pr":                 {"value": 0.8422, "threshold": 0.72, "pass": True},
        "recall":                 {"value": 0.82,   "threshold": 0.75, "pass": True},
        "precision_at_threshold": {"value": 0.93,   "threshold": 0.65, "pass": True},
        "f1":                     {"value": 0.8723, "threshold": 0.68, "pass": True},
        "precision_at_75_recall": {"value": 1.00,   "threshold": 0.65, "pass": True}
    },
    "poc_gate_overall": "PASS"
}

with open(r"../data/processed\poc_evaluation_results.json", "w") as f:
    json.dump(poc_results, f, indent=2)

print("Saved: poc_evaluation_results.json")


Saved: poc_evaluation_results.json
